# 00-02 텐서 기초

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import torch
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 텐서 만들기

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])  # (3,)
b = torch.tensor([[1, 2, 3], [4, 5, 6]])  # (2, 3)
print(a)
print(b)
print(a.shape, b.shape)
print(a.dtype, b.dtype)

In [ ]:
zeros = torch.zeros(2, 3)  # (2, 3)
ones = torch.ones(2, 3)  # (2, 3)
rand = torch.rand(2, 3)  # (2, 3), 0~1 사이 무작위 값
seq = torch.arange(6)  # (6,)
print(zeros)
print(rand)
print(seq)

In [ ]:
x = torch.arange(12).reshape(3, 4)  # (3, 4)

plt.figure(figsize=(4, 3))
plt.imshow(x, cmap="Blues")
for i in range(3):
    for j in range(4):
        plt.text(j, i, int(x[i, j]), ha="center", va="center")
plt.xticks(range(4))  # 눈금을 칸 번호에만 둬요
plt.yticks(range(3))
plt.xlabel("dim 1 (4)")
plt.ylabel("dim 0 (3)")
plt.title("shape (3, 4)")
plt.colorbar()
plt.show()

### 차원마다 담기는 데이터

1차원은 소리, 2차원은 흑백 이미지, 3차원은 컬러 이미지, 4차원은 영상이에요.

In [ ]:
sr = 8000  # 1초에 8000번 기록해요
t = torch.arange(sr) / sr  # (8000,) 0초부터 1초까지의 시각
sound = torch.sin(2 * torch.pi * 440 * t) + 0.5 * torch.sin(2 * torch.pi * 880 * t)  # (8000,)

plt.figure(figsize=(6, 2.5))
plt.plot(t[:160], sound[:160])  # 앞쪽 0.02초만 그려요
plt.xlabel("time (s)")
plt.ylabel("amplitude")
plt.title(f"sound, shape {tuple(sound.shape)}")
plt.show()

In [ ]:
from IPython.display import Audio

Audio(sound.numpy(), rate=sr)  # 재생 버튼을 눌러 들어 보세요

In [ ]:
from matplotlib import cbook

photo = torch.tensor(plt.imread(cbook.get_sample_data("grace_hopper.jpg")))  # (600, 512, 3)
color = photo.permute(2, 0, 1).float() / 255  # (3, 600, 512), 0~1 사이 값
gray = color.mean(dim=0)  # (600, 512), 세 채널의 평균
print(photo.shape, color.shape, gray.shape)

fig, axes = plt.subplots(1, 2, figsize=(7, 4))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title(f"grayscale, shape {tuple(gray.shape)}")
axes[1].imshow(color.permute(1, 2, 0))  # matplotlib은 (높이, 너비, 채널) 순서로 그려요
axes[1].set_title(f"color, shape {tuple(color.shape)}")
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
video = torch.stack([color[:, 60 * i : 60 * i + 300, 100:400] for i in range(4)])  # (4, 3, 300, 300)
print(video.shape)

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for i, ax in enumerate(axes):
    ax.imshow(video[i].permute(1, 2, 0))  # i번째 프레임, (300, 300, 3)
    ax.set_title(f"frame {i}")
    ax.axis("off")
fig.suptitle(f"video, shape {tuple(video.shape)}")
plt.show()

### 차원이 같아도 그리는 방법은 달라요

In [ ]:
scores = (torch.randn(100) * 10 + 70).clamp(0, 100)  # (100,) 학생 100명의 점수
points = torch.randn(200, 2)  # (200, 2) 평면 위의 점 200개

fig, axes = plt.subplots(2, 2, figsize=(8, 7))
axes[0, 0].plot(t[:160], sound[:160])
axes[0, 0].set_title(f"1D sound {tuple(sound.shape)}: line")
axes[0, 1].hist(scores, bins=10)
axes[0, 1].set_title(f"1D scores {tuple(scores.shape)}: histogram")
axes[1, 0].imshow(gray, cmap="gray")
axes[1, 0].set_title(f"2D image {tuple(gray.shape)}: image")
axes[1, 0].axis("off")
axes[1, 1].scatter(points[:, 0], points[:, 1], s=10)
axes[1, 1].set_title(f"2D points {tuple(points.shape)}: scatter")
axes[1, 1].set_aspect("equal")
plt.tight_layout()
plt.show()

## 2. 모양 바꾸기

In [ ]:
x = torch.arange(12)  # (12,)
print(x.reshape(3, 4).shape)  # (3, 4)
print(x.reshape(2, 2, 3).shape)  # (2, 2, 3)
print(x.reshape(4, -1).shape)  # (4, 3), -1은 남는 크기를 자동으로 계산해요

In [ ]:
m = torch.arange(6).reshape(2, 3)  # (2, 3)
print(m.T.shape)  # (3, 2)
print(m.unsqueeze(0).shape)  # (1, 2, 3)
print(m.unsqueeze(0).squeeze(0).shape)  # (2, 3)

## 3. 인덱싱

In [ ]:
m = torch.arange(12).reshape(3, 4)  # (3, 4)
print(m[0])  # 첫 번째 행, (4,)
print(m[:, 1])  # 두 번째 열, (3,)
print(m[1:, 2:])  # (2, 2)

### 조건으로 거르기

`m > 6`은 위치가 아니라 `m`과 모양이 같은 참/거짓 텐서(마스크)예요.

In [ ]:
mask = m > 6  # (3, 4), 칸마다 True 또는 False
print(mask)
print(m[mask])  # True인 칸의 값만, (5,)
print(m[m > 6])  # 두 줄을 합친 형태, 결과는 같아요

## 4. 연산

In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])  # (2, 2)
print(x + 10)
print(x * x)  # 같은 위치끼리 곱하기
print(x.sum(), x.mean())
print(x.sum(dim=0))  # 0번 축을 따라 더하기, (2,)
print(x.sum(dim=1))  # 1번 축을 따라 더하기, (2,)

In [ ]:
A = torch.rand(4, 3)  # (4, 3)
B = torch.rand(3, 2)  # (3, 2)
C = A @ B  # (4, 2)
print(C.shape)

## 5. 브로드캐스팅

In [ ]:
x = torch.zeros(4, 3)  # (4, 3)
bias = torch.tensor([1.0, 2.0, 3.0])  # (3,)
print(x + bias)  # bias가 4개 행에 똑같이 더해져요, (4, 3)

## 6. GPU로 옮기기

In [ ]:
x = torch.rand(3, 3)  # (3, 3)
print(x.device)
x_gpu = x.to(device)
print(x_gpu.device)

In [ ]:
y = torch.rand(3, 3)  # CPU에 있는 텐서
try:
    x_gpu + y
    print("같은 장치에 있어서 계산돼요")
except RuntimeError as e:
    print("오류:", str(e)[:80])

In [ ]:
import time

def time_matmul(dev, n=2048, repeat=10):
    a = torch.rand(n, n, device=dev)  # (n, n)
    b = torch.rand(n, n, device=dev)  # (n, n)
    a @ b  # 첫 실행은 준비 시간이 섞여서 한 번 버려요
    if dev == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(repeat):
        a @ b
    if dev == "cuda":
        torch.cuda.synchronize()
    return (time.time() - start) / repeat

print(f"CPU: {time_matmul('cpu') * 1000:.1f} ms")
if device == "cuda":
    print(f"GPU: {time_matmul('cuda') * 1000:.1f} ms")

## 7. 파이썬 값과 NumPy로 바꾸기

In [ ]:
s = torch.tensor([1.0, 2.0, 3.0]).sum()
print(s, s.item())

g = torch.rand(2, 2).to(device)
print(g.cpu().numpy())